For the segmentation training downloading a dataset with GT labels, is needed.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models.segmentation import deeplabv3_resnet101, DeepLabV3_ResNet101_Weights
import matplotlib.pyplot as plt
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ============ CONFIG ============
ENDOVIS2017_ROOT = Path("./data/endovis2017")

IMG_SIZE = 512
BATCH_SIZE = 4
EPOCHS = 50
LR = 1e-4

CHECKPOINT_OUT = Path("deeplabv3_instrument.pt")

# ============ EARLY STOPPING ============
EARLY_STOPPING_PATIENCE = 10


# ============ DATASET ============
class Endovis2017Dataset(Dataset):

    def __init__(self, root_dir, img_size=IMG_SIZE, train=False):
        self.root_dir = Path(root_dir)
        self.img_size = img_size
        self.train = train

        self.img_dir = self.root_dir / "image"
        self.label_dir = self.root_dir / "label"

        if not self.img_dir.exists() or not self.label_dir.exists():
            raise FileNotFoundError(f"Missing image or label folder in {root_dir}")

        self.img_files = sorted(self.img_dir.glob("seq*_frame*.bmp"))
        self.img_files.extend(sorted(self.img_dir.glob("seq*_frame*.BMP")))

        self.valid_pairs = []
        for img_path in self.img_files:
            label_path = self.label_dir / f"{img_path.stem}.bmp"
            if label_path.exists():
                self.valid_pairs.append((img_path, label_path))

        print(f"  Loaded {len(self.valid_pairs)} pairs from {root_dir}")

        # Base transforms (always applied)
        self.base_transform = T.Compose([
            T.Resize((img_size, img_size)),
        ])

        self.to_tensor_normalize = T.Compose([
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.valid_pairs)

    def __getitem__(self, idx):
        img_path, label_path = self.valid_pairs[idx]

        img = Image.open(img_path).convert("RGB")
        mask = Image.open(label_path).convert("L")
        mask = mask.resize((self.img_size, self.img_size), Image.NEAREST)
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)

        if self.train:
            # Horizontal flip
            if np.random.rand() < 0.5:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
                mask = mask.transpose(Image.FLIP_LEFT_RIGHT)

            # Vertical flip
            if np.random.rand() < 0.3:
                img = img.transpose(Image.FLIP_TOP_BOTTOM)
                mask = mask.transpose(Image.FLIP_TOP_BOTTOM)

            # Random rotation
            angle = np.random.uniform(-20, 20)
            img = img.rotate(angle, resample=Image.BILINEAR)
            mask = mask.rotate(angle, resample=Image.NEAREST)

            # Color jitter (image only, not mask)
            color_jitter = T.ColorJitter(
                brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05
            )
            img = color_jitter(img)

        img_t = self.to_tensor_normalize(img)
        mask_t = torch.from_numpy((np.array(mask) > 0).astype(np.float32))

        return img_t, mask_t


def find_endovis2017_folders(root):
    folders = []
    for item in root.iterdir():
        if item.is_dir():
            if item.name == "train" or item.name.startswith("val"):
                if (item / "image").exists() and (item / "label").exists():
                    folders.append(item)
    return sorted(folders)


# ============ LOSSES ============
def dice_loss(pred_logits, target, eps=1e-6):
    pred = torch.sigmoid(pred_logits)
    pred_flat = pred.flatten(1)
    target_flat = target.flatten(1)
    intersection = (pred_flat * target_flat).sum(dim=1)
    union = pred_flat.sum(dim=1) + target_flat.sum(dim=1)
    dice = (2 * intersection + eps) / (union + eps)
    return 1 - dice.mean()


def build_model():
    weights = DeepLabV3_ResNet101_Weights.DEFAULT
    model = deeplabv3_resnet101(weights=weights)
    model.classifier[4] = nn.Conv2d(256, 1, kernel_size=1)
    if model.aux_classifier is not None:
        model.aux_classifier[4] = nn.Conv2d(256, 1, kernel_size=1)
    return model.to(device)


def visualize_prediction(model, val_loader, num_samples=4):
    model.eval()
    imgs, masks = next(iter(val_loader))
    imgs, masks = imgs.to(device), masks.to(device)

    with torch.no_grad():
        logits = model(imgs)["out"].squeeze(1)
        preds = torch.sigmoid(logits) > 0.5

    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)
    imgs_vis = torch.clamp(imgs * std + mean, 0, 1)

    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))
    for i in range(num_samples):
        axes[i, 0].imshow(imgs_vis[i].cpu().permute(1, 2, 0))
        axes[i, 0].set_title("Input Image")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(masks[i].cpu(), cmap="gray")
        axes[i, 1].set_title("Ground Truth")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(preds[i].cpu(), cmap="gray")
        axes[i, 2].set_title("Prediction")
        axes[i, 2].axis("off")

    plt.tight_layout()
    plt.show()


# ============ MAIN ============
print("=" * 60)
print("DeepLabV3+ Training for Endovis2017 Instrument Segmentation")
print("=" * 60)

print("\nBuilding DeepLabv3+ (ResNet101 backbone, COCO-pretrained)...")
model = build_model()
print(f"  ✅ Model built with {sum(p.numel() for p in model.parameters()):,} parameters")

print(f"\nScanning: {ENDOVIS2017_ROOT}")
if not ENDOVIS2017_ROOT.exists():
    print(f"❌ ERROR: Path not found: {ENDOVIS2017_ROOT}")
else:
    print("\nAll folders in root:")
    for item in sorted(ENDOVIS2017_ROOT.iterdir()):
        if item.is_dir():
            print(f"  {item.name}")

    folders = find_endovis2017_folders(ENDOVIS2017_ROOT)
    print(f"\nFound {len(folders)} folders with image/label structure:")

    train_folders = [f for f in folders if f.name == "train"]

    val_folders = [f for f in folders if f.name.startswith("val")]
    val_folders_sorted = sorted(val_folders, key=lambda x: int(x.name.replace("val", "")))
    val_folders_final = [f for f in val_folders_sorted if f.name == "val10"]
    
    train_extra      = [f for f in val_folders_sorted if f.name not in ("val9", "val10")]
    train_folders    = train_folders + train_extra

    print(f"\n  Train folders: {[f.name for f in train_folders]}")
    print(f"  Val folders:   {[f.name for f in val_folders_final]}")
    print(f"  Excluded:      val9 (only 1 pair)")
    # ─────────────────────────────────────────────────────────────

    if not train_folders:
        print("\n❌ No train folder found! Check ENDOVIS2017_ROOT path.")
    else:
        print("\n" + "-" * 60)
        print("Loading datasets...")
        print("-" * 60)

        train_ds = Endovis2017Dataset(train_folders[0], train=True)
        for folder in train_folders[1:]:
            extra_ds = Endovis2017Dataset(folder, train=True)
            train_ds.valid_pairs.extend(extra_ds.valid_pairs)

        val_ds = Endovis2017Dataset(val_folders_final[0], train=False)

        print(f"\n✅ Total train pairs: {len(train_ds)}")
        print(f"✅ Total val pairs:   {len(val_ds)}")

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

        optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

        best_val_iou = 0.0
        best_val_dice = 0.0
        epochs_since_improvement = 0

        history = {'train_loss': [], 'val_iou': [], 'val_dice': []}

        print("\n" + "=" * 60)
        print("Starting Training  |  Loss: Dice + BCE")
        print("=" * 60)

        for epoch in range(EPOCHS):
            # ── Train ──
            model.train()
            train_loss = 0.0

            loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
            for imgs, masks in loop:
                imgs, masks = imgs.to(device), masks.to(device)

                out         = model(imgs)
                main_logits = out["out"].squeeze(1)
                loss        = dice_loss(main_logits, masks) + F.binary_cross_entropy_with_logits(main_logits, masks)

                if "aux" in out:
                    aux_logits = out["aux"].squeeze(1)
                    loss = loss + 0.4 * (
                        dice_loss(aux_logits, masks) + F.binary_cross_entropy_with_logits(aux_logits, masks)
                    )

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                train_loss += loss.item() * imgs.size(0)
                loop.set_postfix(loss=loss.item())

            train_loss /= len(train_ds)
            scheduler.step()

            # ── Validation ──
            model.eval()
            val_iou_sum, val_dice_sum, val_count = 0.0, 0.0, 0

            with torch.no_grad():
                for imgs, masks in val_loader:
                    imgs, masks = imgs.to(device), masks.to(device)
                    logits = model(imgs)["out"].squeeze(1)
                    pred   = (torch.sigmoid(logits) > 0.5).float()

                    intersection = (pred * masks).sum(dim=(1, 2))
                    union        = ((pred + masks) > 0).float().sum(dim=(1, 2))
                    iou  = torch.where(union > 0, intersection / union,        torch.zeros_like(union))
                    pred_sum  = pred.sum(dim=(1, 2))
                    mask_sum  = masks.sum(dim=(1, 2))
                    dice = torch.where(union > 0, (2 * intersection) / (pred_sum + mask_sum), torch.zeros_like(union))

                    val_iou_sum  += iou.sum().item()
                    val_dice_sum += dice.sum().item()
                    val_count    += imgs.size(0)

            val_iou  = val_iou_sum  / max(val_count, 1)
            val_dice = val_dice_sum / max(val_count, 1)

            history['train_loss'].append(train_loss)
            history['val_iou'].append(val_iou)
            history['val_dice'].append(val_dice)

            print(f"Epoch [{epoch+1:2d}/{EPOCHS}]  |  Loss: {train_loss:.4f}  |  Val IoU: {val_iou:.4f}  |  Val Dice: {val_dice:.4f}")

            # ── Check improvement ──
            if val_iou > best_val_iou:
                best_val_iou = val_iou
                best_val_dice = val_dice
                epochs_since_improvement = 0
                torch.save({
                    'epoch': epoch + 1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_iou': val_iou,
                    'val_dice': val_dice,
                    'history': history
                }, CHECKPOINT_OUT)
                print(f"  ✅ New best! Saved checkpoint (IoU: {val_iou:.4f}, Dice: {val_dice:.4f})")
            else:
                epochs_since_improvement += 1
                print(f"  No improvement for {epochs_since_improvement} epoch(s)")

            # ── Early stopping ──
            if epochs_since_improvement >= EARLY_STOPPING_PATIENCE:
                print(f"\n🛑 Early stopping: no improvement in {EARLY_STOPPING_PATIENCE} epochs "
                    f"(best was epoch {epoch + 1 - epochs_since_improvement}, IoU={best_val_iou:.4f})")
                break

            if (epoch + 1) % 5 == 0:
                print("\nVisualizing predictions...")
                visualize_prediction(model, val_loader, num_samples=3)

        # ── Summary ──
        print("\n" + "=" * 60)
        print("TRAINING COMPLETE")
        print("=" * 60)
        print(f"Best validation IoU:  {best_val_iou:.4f}")
        print(f"Best validation Dice: {best_val_dice:.4f}")
        print(f"Checkpoint saved to:  {CHECKPOINT_OUT}")

        # ── Plot ──
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(history['train_loss'])
        axes[0].set_title('Training Loss')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].grid(True)

        axes[1].plot(history['val_iou'],  label='IoU')
        axes[1].plot(history['val_dice'], label='Dice')
        axes[1].set_title('Validation Metrics')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Score')
        axes[1].legend()
        axes[1].grid(True)

        plt.tight_layout()
        plt.show()

        print("\nFinal predictions on validation set:")
        visualize_prediction(model, val_loader, num_samples=4)

        # ── Evaluate checkpoint ──
        print("\n" + "=" * 60)
        print("EVALUATING CHECKPOINT")
        print("=" * 60)
        if CHECKPOINT_OUT.exists():
            checkpoint = torch.load(CHECKPOINT_OUT, map_location=device)
            print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
            print(f"  Val IoU:  {checkpoint['val_iou']:.4f}")
            print(f"  Val Dice: {checkpoint['val_dice']:.4f}")
        else:
            print(f"❌ Checkpoint not found: {CHECKPOINT_OUT}")